# Aula 27: Árvores de Decisão (Decision Trees) — Diagnóstico Preditivo e Classificação Operacional do Visão-AGV

## 1. Visão Geral da Aula

Nesta aula, implementaremos um modelo clássico de **Árvore de Decisão** supervisionada baseada no cálculo de **Entropia e Ganho de Informação** (Algoritmo ID3) para realizar o diagnóstico de saúde operacional e classificação do estado do **Visão-AGV**.

### Contexto no Projeto Visão-AGV
Para garantir a manutenção preditiva e evitar falhas no meio da rota intralogística, o sistema SCADA monitora continuamente variáveis de telemetria da frota, tais como:
- `nivel_bateria`: Normal vs. Baixa
- `temperatura_motor`: Normal vs. Alta
- `ruido_sensor_lidar`: Baixo vs. Alto

O objetivo da Árvore de Decisão é aprender com o histórico de telemetria e classificar a ação operacional recomendada para o AGV:
- `OPERACAO_NORMAL`
- `RECARGA_AGENDADA`
- `MANUTENCAO_IMEDIATA`

---

In [ ]:
# Implementação de Métricas de Entropia e Ganho de Informação
import math
from collections import Counter

def calcular_entropia(rotulos):
    """
    Calcula a Entropia de Shannon H(S) de um conjunto de rótulos de classe.
    """
    total = len(rotulos)
    if total == 0:
        return 0.0
    
    contagem = Counter(rotulos)
    entropia = 0.0
    for count in contagem.values():
        p = count / total
        entropia -= p * math.log2(p)
    return entropia

def calcular_ganho_informacao(dados, atributo_idx, rotulos):
    """
    Calcula o Ganho de Informação IG(S, A) ao dividir os dados por um determinado atributo.
    """
    entropia_total = calcular_entropia(rotulos)
    valores_atributo = set(row[atributo_idx] for row in dados)
    
    entropia_ponderada = 0.0
    total = len(dados)
    
    for val in valores_atributo:
        sub_rotulos = [rotulos[i] for i in range(total) if dados[i][atributo_idx] == val]
        p = len(sub_rotulos) / total
        entropia_ponderada += p * calcular_entropia(sub_rotulos)
        
    return entropia_total - entropia_ponderada

print("Funções de Entropia e Ganho de Informação declaradas com sucesso!")

## 2. Construção Recursiva da Árvore de Decisão (Algoritmo ID3)

A estrutura da árvore escolhe iterativamente o atributo com maior Ganho de Informação para dividir o conjunto de dados até que as folhas sejam puras ou os atributos se esgotem.

In [ ]:
class NoDecisao:
    """
    Representa um nó interno (teste de atributo) ou folha (decisão final) na árvore.
    """
    def __init__(self, atributo=None, valor=None, resultado=None):
        self.atributo = atributo    # Nome do atributo testado
        self.resultado = resultado  # Rótulo da folha (se for nó folha)
        self.filhos = {}          # Dicionário {valor_atributo: NoDecisao}

def construir_arvore_id3(dados, nomes_atributos, rotulos):
    # Caso Base 1: Todos os exemplos pertencem à mesma classe
    if len(set(rotulos)) == 1:
        return NoDecisao(resultado=rotulos[0])
    
    # Caso Base 2: Não há mais atributos para testar (votação majoritária)
    if not nomes_atributos or len(dados[0]) == 0:
        classe_mais_comum = Counter(rotulos).most_common(1)[0][0]
        return NoDecisao(resultado=classe_mais_comum)
    
    # Escolha do melhor atributo com base no maior Ganho de Informação
    ganho_info = [calcular_ganho_informacao(dados, i, rotulos) for i in range(len(nomes_atributos))]
    melhor_idx = ganho_info.index(max(ganho_info))
    melhor_atributo = nomes_atributos[melhor_idx]
    
    raiz = NoDecisao(atributo=melhor_atributo)
    valores_possiveis = set(row[melhor_idx] for row in dados)
    
    for val in valores_possiveis:
        sub_dados = []
        sub_rotulos = []
        for i in range(len(dados)):
            if dados[i][melhor_idx] == val:
                # Remove a coluna do atributo selecionado para a recursão
                nova_linha = dados[i][:melhor_idx] + dados[i][melhor_idx+1:]
                sub_dados.append(nova_linha)
                sub_rotulos.append(rotulos[i])
                
        novos_atributos = nomes_atributos[:melhor_idx] + nomes_atributos[melhor_idx+1:]
        raiz.filhos[val] = construir_arvore_id3(sub_dados, novos_atributos, sub_rotulos)
        
    return raiz

def predizer(arvore, amostra, nomes_atributos):
    """Classifica uma nova amostra de telemetria percorrendo a árvore."""
    if arvore.resultado is not None:
        return arvore.resultado
    
    idx_attr = nomes_atributos.index(arvore.atributo)
    val_amostra = amostra[idx_attr]
    
    if val_amostra in arvore.filhos:
        return predizer(arvore.filhos[val_amostra], amostra, nomes_atributos)
    else:
        # Retorno de fallback caso encontre valor não visto no treino
        return "DESCONHECIDO"

print("Algoritmo ID3 e função de predição prontos!")

## 3. Simulação Prática: Treinamento e Diagnóstico no Visão-AGV

In [ ]:
# Conjunto de Dados Histórico de Telemetria (Treino)
# Atributos: ['Nivel_Bateria', 'Temperatura_Motor', 'Ruido_LiDAR']
nomes_atributos = ['Nivel_Bateria', 'Temperatura_Motor', 'Ruido_LiDAR']

X_treino = [
    ['Alta', 'Normal', 'Baixo'],
    ['Alta', 'Alta',   'Baixo'],
    ['Baixa', 'Normal', 'Baixo'],
    ['Alta', 'Normal', 'Alto'],
    ['Baixa', 'Alta',   'Alto'],
    ['Alta', 'Alta',   'Alto'],
    ['Baixa', 'Normal', 'Alto']
]

y_treino = [
    'OPERACAO_NORMAL',
    'MANUTENCAO_IMEDIATA',
    'RECARGA_AGENDADA',
    'OPERACAO_NORMAL',
    'MANUTENCAO_IMEDIATA',
    'MANUTENCAO_IMEDIATA',
    'RECARGA_AGENDADA'
]

# Treinamento da Árvore de Decisão
arvore_agv = construir_arvore_id3(X_treino, nomes_atributos, y_treino)

# Teste com Novas Leituras do SCADA em Tempo Real
novas_leituras = [
    ['Alta', 'Normal', 'Baixo'],  # AGV-01
    ['Baixa', 'Normal', 'Baixo'], # AGV-02
    ['Alta', 'Alta', 'Baixo']     # AGV-03
]

print("=== PREDIÇÃO DE DIAGNÓSTICO SCADA VIA ÁRVORE DE DECISÃO ===\n")
for idx, amostra in enumerate(novas_leituras, 1):
    resultado = predizer(arvore_agv, amostra, nomes_atributos)
    print(f"AGV-{idx:02d} {amostra} -> Ação Recomendada: {resultado}")

## 4. Análise de Resultados e Conclusão

1. **Determinismo e Transparência:** Diferente de modelos de caixa-preta, as Árvores de Decisão permitem que os engenheiros do SCADA inspecionem as regras lógicas exatas de decisão.
2. **Automatização Preditiva:** O modelo classifica com facilidade falhas operacionais térmicas ou de alimentação antes que ocorram paradas não planejadas.
3. **Conclusão da Etapa 03:** Com a conclusão da Aula 27, toda a base teórica e prática sobre Árvores (MSTs, Huffman, BST, RRT, STP/RSTP, Redes de Petri, Behavior Trees, KD-Trees e Árvores de Decisão) está consolidada para o **Projeto Visão-AGV**!